<a href="https://colab.research.google.com/github/nika19du/AI-for-Developers-summer-2026-/blob/main/ContextManagement(Summarization_middleware).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install -q langchain langchain-openai

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 125.1/125.1 kB 3.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 570.0/570.0 kB 17.5 MB/s eta 0:00:00


In [2]:
from google.colab import userdata
from langchain.agents import AgentState, create_agent
from langchain.agents.middleware import SummarizationMiddleware, after_model
from langchain_core.messages import HumanMessage
from langchain_core.messages import BaseMessage
from langchain_core.runnables import RunnableLambda
from langchain_openai import ChatOpenAI
from langgraph.checkpoint.memory import InMemorySaver
from langgraph.runtime import Runtime
from pydantic import SecretStr
from typing import Annotated, List

api_key = userdata.get('OPENAI_API_KEY')

def print_conversation(conversation: List[BaseMessage]):
    for message in conversation:
        message.pretty_print()

middleware e конфигуриран така че в момента в който имаме 5 съобщения той се активира и последните две съобщения ще считаме за актуални така че не трябва да участват в обобщението.
Не винаги middleware ще се придържа към стойностите които сме подали
той ще направи това което счита най-опитмално спрямо структурата на контекста
```
 middleware = [
        SummarizationMiddleware( # обобщаване на старата информация, защото ai models имат ограничен контекст. Колкото повече запълваме този контекст толкова по неефективен става този модел
            model = model, # с кой модел ще се изпълни обобщението на по старата част от итеракцията с агента
            trigger = [("messages", 5)], # trigger: това е условието, което трябва да бъде изпълнено, за да активираме middleware
            keep = ("messages", 2) # keep: казва какво ние считаме за актуално.
        )
    ]```



In [11]:
model = ChatOpenAI(model = "gpt-5-mini", api_key = api_key, reasoning_effort="low")

agent = create_agent(
    model = model,
    system_prompt = "You are caring teaching assistant specializing in mathematics and computer sciences. Provide clear and detailed explanations to every question.",
    checkpointer = InMemorySaver(),
    middleware = [
        SummarizationMiddleware( # обобщаване на старата информация, защото ai models имат ограничен контекст. Колкото повече запълваме този контекст толкова по неефективен става този модел
            model = model, # с кой модел ще се изпълни обобщението на по старата част от итеракцията с агента
            trigger = [("messages", 5)], # trigger: това е условието, което трябва да бъде изпълнено, за да активираме middleware
            keep = ("messages", 2) # keep: казва какво ние считаме за актуално.
        )
    ]
)

interact = agent | RunnableLambda(lambda res: print_conversation(res["messages"]))

In [12]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Prove that the square root of 2 is irrational.")]
    },
    config = {
        "configurable": {
            "thread_id": "math_proof"
        }
    }
)

================================ Human Message =================================

Prove that the square root of 2 is irrational.
================================== Ai Message ==================================

Proof by contradiction (classic parity argument).

Assume, for contradiction, that √2 is rational. Then there exist integers p and q (q ≠ 0) with gcd(p, q) = 1 such that
√2 = p/q.
Square both sides:
2 = p^2/q^2, so p^2 = 2q^2.

Thus p^2 is even, so p must be even. (Reason: if p were odd, p = 2k+1, then p^2 = 4k(k+1)+1 would be odd; so p^2 even implies p even.) Write p = 2r for some integer r. Substitute into p^2 = 2q^2:
(2r)^2 = 2q^2 ⇒ 4r^2 = 2q^2 ⇒ 2r^2 = q^2.

So q^2 is even, hence q is even by the same parity reasoning.

We have deduced that both p and q are even, so they share a factor 2. That contradicts the assumption that gcd(p, q) = 1 (i.e., that the fraction p/q was in lowest terms).

Therefore our original assumption was false, and √2 is irrational.

(Alternative viewp

In [13]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Prove that the sum of all angles in a given triangle equals 180 degrees.")]
    },
    config = {
        "configurable": {
            "thread_id": "math_proof"
        }
    }
)

================================ Human Message =================================

Prove that the square root of 2 is irrational.
================================== Ai Message ==================================

Proof by contradiction (classic parity argument).

Assume, for contradiction, that √2 is rational. Then there exist integers p and q (q ≠ 0) with gcd(p, q) = 1 such that
√2 = p/q.
Square both sides:
2 = p^2/q^2, so p^2 = 2q^2.

Thus p^2 is even, so p must be even. (Reason: if p were odd, p = 2k+1, then p^2 = 4k(k+1)+1 would be odd; so p^2 even implies p even.) Write p = 2r for some integer r. Substitute into p^2 = 2q^2:
(2r)^2 = 2q^2 ⇒ 4r^2 = 2q^2 ⇒ 2r^2 = q^2.

So q^2 is even, hence q is even by the same parity reasoning.

We have deduced that both p and q are even, so they share a factor 2. That contradicts the assumption that gcd(p, q) = 1 (i.e., that the fraction p/q was in lowest terms).

Therefore our original assumption was false, and √2 is irrational.

(Alternative viewp

In [14]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Prove that the square root of 5 is irrational.")]
    },
    config = {
        "configurable": {
            "thread_id": "math_proof"
        }
    }
)

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user requests elementary mathematical proofs. Specifically: (1) prove that √2 is irrational, and (2) prove that the sum of the interior angles of a triangle equals 180°.

## SUMMARY
- User asked: "Prove that the square root of 2 is irrational."
- Assistant provided a classic proof by contradiction using parity:
  - Assume √2 = p/q with integers p, q in lowest terms (gcd(p,q)=1).
  - Square: p^2 = 2q^2, so p^2 is even ⇒ p is even. Write p = 2r.
  - Substitute: 4r^2 = 2q^2 ⇒ q^2 = 2r^2, so q^2 is even ⇒ q is even.
  - Both p and q even contradict gcd(p,q)=1. Hence √2 is irrational.
  - Alternative viewpoint mentioned: infinite descent / well-ordering contradiction.
- User then asked: "Prove that the sum of all angles in a given triangle equals 180 degrees."
  - No proof for the triangle angle sum has been given yet; this is the outstandin

In [15]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Prove that there are infinitely many prime numbers.")]
    },
    config = {
        "configurable": {
            "thread_id": "math_proof"
        }
    }
)

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user requests elementary mathematical proofs. Specifically:
- Earlier: prove that √2 is irrational, and prove that the sum of the interior angles of a triangle equals 180°.
- Current/new request: prove that √5 is irrational.

## SUMMARY
- The user originally asked for a proof that √2 is irrational.
  - Assistant provided the standard proof by contradiction using parity: assume √2 = p/q in lowest terms ⇒ p^2 = 2q^2 ⇒ p even ⇒ q even ⇒ contradiction with gcd(p,q)=1. Alternative viewpoint (infinite descent / well-ordering) was mentioned.
- The user asked for a proof that the sum of the angles in a triangle equals 180°.
  - Assistant provided a Euclidean proof using a line through one vertex parallel to the opposite side and the alternate interior angles theorem, concluding ∠A + ∠B + ∠C = 180°. It was noted this uses the Euclidean parallel 

In [16]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Prove the cosine theorem.")]
    },
    config = {
        "configurable": {
            "thread_id": "math_proof"
        }
    }
)

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to obtain elementary mathematical proofs. Specifically, earlier requests were to prove that √2 is irrational and that the sum of interior angles of a triangle equals 180°. The user then requested a proof that √5 is irrational (which the assistant provided). The current outstanding request is: "Prove that there are infinitely many prime numbers."

## SUMMARY
- Requested proofs:
  - √2 is irrational.
    - Assistant provided the standard proof by contradiction using parity: assume √2 = p/q in lowest terms ⇒ p^2 = 2q^2 ⇒ p even ⇒ q even ⇒ contradiction with gcd(p,q)=1. Alternative viewpoints (infinite descent / well-ordering) noted.
  - Sum of interior angles of a triangle equals 180°.
    - Assistant provided a Euclidean proof: draw a line through one vertex parallel to the opposite side; use alternate interior angl

In [17]:
interact.invoke(
    input = {
        "messages": [HumanMessage("Show me how to solve quadratic equations. Then, explain if this approach is precise when implemented in a software.")]
    },
    config = {
        "configurable": {
            "thread_id": "math_proof"
        }
    }
)

================================ Human Message =================================

Here is a summary of the conversation to date:

## SESSION INTENT
The user's primary goal is to obtain elementary mathematical proofs. They previously requested several classical proofs (irrationality of √2 and √5, sum of triangle angles = 180°) and now have asked: "Prove the cosine theorem" (the law of cosines).

## SUMMARY
- Previously requested and obtained proofs:
  - √2 is irrational: standard contradiction using parity (or infinite descent).
  - Sum of interior angles of a triangle equals 180°: Euclidean proof using a line through a vertex parallel to the opposite side; relies on the parallel postulate.
  - √5 is irrational: contradiction using divisibility by 5; generalizes to √n for non-square integers via prime divisibility.
  - Infinitely many primes: Euclid's proof by contradiction using N = product of listed primes + 1; other proofs (Euler analytic, Dirichlet, Erdős) were noted but not require